In [1]:
pip install groq

Note: you may need to restart the kernel to use updated packages.


In [16]:
# Cell 1 - Setup
from dotenv import load_dotenv
from groq import Groq
import os
import json
from statistics import mean

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
model = "llama-3.3-70b-versatile"

In [17]:
# Helper functions
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system=None, temperature=1.0, stop=None):
    all_messages = []
    if system:
        all_messages.append({"role": "system", "content": system})
    all_messages += messages

    response = client.chat.completions.create(
        model=model,
        max_tokens=1000,
        messages=all_messages,
        temperature=temperature,
        stop=stop,
    )
    return response.choices[0].message.content

In [52]:
#Generate dataset
def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop=["```"])
    return json.loads(text)

In [53]:
# Save dataset
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)
print(json.dumps(dataset, indent=2))

[
  {
    "task": "Create a JSON object representing an AWS S3 bucket policy that allows read-only access to a specific bucket.",
    "format": "json",
    "solution_criteria": "The policy should specify the correct bucket name, allow the 's3:GetObject' action, and include a statement ID."
  },
  {
    "task": "Write a Python function that takes an AWS IAM policy document as input and returns a list of all the actions specified in the policy.",
    "format": "python",
    "solution_criteria": "The function should handle policies with multiple statements and return a list of unique action names."
  },
  {
    "task": "Create a regular expression that matches the format of an AWS ARN (Amazon Resource Name) for an S3 bucket.",
    "format": "regex",
    "solution_criteria": "The regular expression should match the 'arn:aws:s3:::bucket_name' format, where 'bucket_name' can be any valid S3 bucket name."
  }
]


In [ ]:
# Grade output using model 
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>


Provide your evaluation as a JSON object with these fields:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation (do NOT use backslashes in this field)
- "score": A number between 1-10

Respond with JSON only. Do not include any backslashes in string values.
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop=["```"])
    
    # Fix invalid escape sequences before parsing
    eval_text = eval_text.replace("\\", "\\\\")   # fix backslashes
    eval_text = eval_text.replace("\\\\/", "/")    # fix over-escaped slashes
    
    try:
        return json.loads(eval_text)
    except json.JSONDecodeError:
        # Fallback: use raw_decode or return default
        print(f"⚠️ JSON parse failed. Raw output:\n{eval_text}")
        return {"strengths": [], "weaknesses": [], "reasoning": "Parse error", "score": 5}

In [56]:
# Run prompt on a test case
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop=["```"])

    # Strip markdown fences if model adds them anyway
    output = output.strip()
    output = re.sub(r"^```[\w]*\n?", "", output)
    output = re.sub(r"\n?```$", "", output)
    return output.strip()

In [57]:
import re
import ast
import json

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [58]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [59]:
# Rul full evals
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [60]:
# Load dataset and run
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 8.0


In [61]:
# Print results
print(json.dumps(results, indent=2))

[
  {
    "output": "{\n  \"Version\": \"2012-10-17\",\n  \"Statement\": [\n    {\n      \"Sid\": \"ReadOnlyAccess\",\n      \"Effect\": \"Allow\",\n      \"Principal\": \"*\",\n      \"Action\": [\n        \"s3:GetObject\",\n        \"s3:ListBucket\"\n      ],\n      \"Resource\": \"arn:aws:s3:::example-bucket\"\n    }\n  ]\n}",
    "test_case": {
      "task": "Create a JSON object representing an AWS S3 bucket policy that allows read-only access to a specific bucket.",
      "format": "json",
      "solution_criteria": "The policy should specify the correct bucket name, allow the 's3:GetObject' action, and include a statement ID."
    },
    "score": 8.0,
    "reasoning": "The solution provides a basic read-only S3 bucket policy but lacks specific access controls and security considerations"
  },
  {
    "output": "import json\n\ndef get_actions(policy_document):\n    policy = json.loads(policy_document)\n    actions = []\n    for statement in policy['Statement']:\n        if 'Actio